In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="trendcast",
    user="postgres",
    password="",
    host="localhost",
    port=5432
)
cur = conn.cursor()
print("Connected!")

Connected!


In [ ]:
conn.rollback()

cur.execute("""
CREATE TABLE IF NOT EXISTS companies (
    company_id      SERIAL PRIMARY KEY,
    company_name    VARCHAR(100) NOT NULL,
    ticker_symbol   VARCHAR(10) UNIQUE NOT NULL,
    industry        VARCHAR(100),
    sector          VARCHAR(100),
    created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS filings (
    filing_id         SERIAL PRIMARY KEY,
    company_id        INT REFERENCES companies(company_id),
    filing_type       VARCHAR(10) NOT NULL,
    filing_date       DATE NOT NULL,
    period_of_report  DATE,
    accession_number  VARCHAR(25) UNIQUE
);

CREATE TABLE IF NOT EXISTS financials (
    financial_id   SERIAL PRIMARY KEY,
    filing_id      INT REFERENCES filings(filing_id),
    company_id     INT REFERENCES companies(company_id),
    revenue        NUMERIC(20, 2),
    net_income     NUMERIC(20, 2),
    eps            NUMERIC(10, 4),
    rd_spend       NUMERIC(20, 2),
    fiscal_year    INT NOT NULL,
    fiscal_quarter INT CHECK (fiscal_quarter BETWEEN 1 AND 4)
);

CREATE TABLE IF NOT EXISTS trend_scores (
    score_id     SERIAL PRIMARY KEY,
    company_id   INT REFERENCES companies(company_id),
    score_date   DATE NOT NULL,
    trend_score  NUMERIC(5, 2),
    keyword      VARCHAR(100),
    source       VARCHAR(50)
);

CREATE TABLE IF NOT EXISTS google_trends (
    id           SERIAL PRIMARY KEY,
    keyword      TEXT,
    timestamp    TEXT,
    search_index INT
);
""")
conn.commit()
print("Tables created!")

Tables created!


In [ ]:
cur.execute("""
INSERT INTO companies (company_name, ticker_symbol, industry, sector) VALUES
    ('Apple Inc.',        'AAPL',   'Consumer Electronics', 'Technology'),
    ('NVIDIA Corp.',      'NVDA',   'Semiconductors',       'Technology'),
    ('Samsung',           'SSNLF',  'Consumer Electronics', 'Technology'),
    ('Sony Group',        'SONY',   'Consumer Electronics', 'Technology'),
    ('Microsoft Corp.',   'MSFT',   'Software',             'Technology'),
    ('Intel Corp.',       'INTC',   'Semiconductors',       'Technology'),
    ('AMD Inc.',          'AMD',    'Semiconductors',       'Technology'),
    ('Logitech',          'LOGI',   'Computer Hardware',    'Technology')
ON CONFLICT (ticker_symbol) DO NOTHING;
""")
conn.commit()
print("Companies inserted!")

Companies inserted!


In [ ]:
cur.execute("DROP VIEW IF EXISTS dashboard_view;")
conn.commit()

cur.execute("""
CREATE VIEW dashboard_view AS
SELECT
    ts.keyword,
    CASE
        WHEN ts.keyword IN (
            'Apple', 'NVIDIA', 'Samsung', 'Logitech', 'Sony',
            'Dell', 'HP', 'Garmin', 'Microsoft', 'Intel', 'AMD'
        ) THEN 'company'
        WHEN ts.keyword IN (
            'smartphone', 'laptop', 'smartwatch', 'tablet',
            'earbuds', 'monitor', 'smart TV'
        ) THEN 'categorical'
        ELSE 'feature'
    END AS keyword_group,
    ts.trend_score,
    ts.score_date AS updated_at,
    ts.source,
    f.revenue,
    f.eps,
    f.net_income,
    NULL::NUMERIC AS sentiment_score,
    NULL::INT     AS news_count,
    NULL::NUMERIC AS google_trends_score
FROM trend_scores ts
LEFT JOIN companies c
    ON LOWER(ts.keyword) = LOWER(c.ticker_symbol)
    OR LOWER(ts.keyword) = LOWER(c.company_name)
LEFT JOIN financials f
    ON f.company_id = c.company_id
LEFT JOIN filings fi
    ON fi.filing_id = f.filing_id
    AND fi.filing_type = '10-K'
ORDER BY ts.score_date DESC;
""")
conn.commit()
print("Dashboard view created!")

Dashboard view created!


In [ ]:
cur.execute("""
CREATE INDEX IF NOT EXISTS idx_filings_company       ON filings(company_id);
CREATE INDEX IF NOT EXISTS idx_financials_company    ON financials(company_id);
CREATE INDEX IF NOT EXISTS idx_financials_year       ON financials(fiscal_year);
CREATE INDEX IF NOT EXISTS idx_trend_scores_date     ON trend_scores(score_date);
CREATE INDEX IF NOT EXISTS idx_trend_scores_keyword  ON trend_scores(keyword);
CREATE INDEX IF NOT EXISTS idx_google_trends_keyword ON google_trends(keyword);
""")
conn.commit()
print("Indexes created!")

Indexes created!


In [ ]:
cur.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public'
""")
print("Tables:", [r[0] for r in cur.fetchall()])

cur.execute("SELECT company_name, ticker_symbol FROM companies")
print("Companies:", cur.fetchall())

Tables: ['companies', 'filings', 'financials', 'trend_scores', 'google_trends', 'dashboard_view']
Companies: [('Apple Inc.', 'AAPL'), ('Sony Group', 'SONY'), ('Dell Technologies', 'DELL'), ('HP Inc.', 'HPQ'), ('Garmin Ltd.', 'GRMN'), ('NVIDIA Corp.', 'NVDA'), ('Microsoft Corp.', 'MSFT'), ('Intel Corp.', 'INTC'), ('AMD Inc.', 'AMD'), ('Logitech', 'LOGI'), ('Samsung', 'SSNLF')]


In [ ]:
# Check all companies are there
cur.execute("SELECT company_id, company_name, ticker_symbol FROM companies ORDER BY company_id")
companies = cur.fetchall()
print("=== COMPANIES ===")
for c in companies:
    print(f"  {c[0]} | {c[1]} | {c[2]}")

# Check all tables exist
cur.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
""")
print("\n=== TABLES ===")
for t in cur.fetchall():
    print(f"  {t[0]}")

# Check dashboard view exists
cur.execute("""
    SELECT table_name FROM information_schema.views
    WHERE table_schema = 'public'
""")
print("\n=== VIEWS ===")
for v in cur.fetchall():
    print(f"  {v[0]}")

=== COMPANIES ===
  1 | Apple Inc. | AAPL
  2 | Sony Group | SONY
  3 | Dell Technologies | DELL
  4 | HP Inc. | HPQ
  5 | Garmin Ltd. | GRMN
  6 | NVIDIA Corp. | NVDA
  10 | Microsoft Corp. | MSFT
  11 | Intel Corp. | INTC
  12 | AMD Inc. | AMD
  13 | Logitech | LOGI
  19 | Samsung | SSNLF

=== TABLES ===
  companies
  dashboard_view
  filings
  financials
  google_trends
  trend_scores

=== VIEWS ===
  dashboard_view
